# Neural ODEs: Continuous Depth Networks

## Introduction

**Neural Ordinary Differential Equations (Neural ODEs)** represent a paradigm shift in how we think about deep learning architectures. Instead of stacking discrete layers, Neural ODEs model transformations as continuous flows defined by differential equations.

**What we'll learn:**
- How residual networks relate to numerical ODE solvers (Euler method)
- The mathematical framework connecting neural networks to differential equations
- How to implement continuous-depth networks using ODE solvers
- The adjoint method for memory-efficient backpropagation
- Practical advantages: adaptive computation, memory efficiency, and continuous representations

**Why it matters:**
Neural ODEs provide an elegant mathematical framework that unifies neural networks with dynamical systems theory. They offer memory efficiency (constant memory vs depth), adaptive computation (solver chooses depth), and smooth interpolation between time points. This connection has profound implications for generative models (continuous normalizing flows), time series modeling, and our theoretical understanding of deep learning.

## Setup

Let's import the necessary libraries. We'll use `torchdiffeq` for ODE solvers, which provides efficient PyTorch implementations of numerical integration methods.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import time

# Try to import torchdiffeq
try:
    from torchdiffeq import odeint_adjoint as odeint
except ImportError:
    print("Installing torchdiffeq...")
    import subprocess
    subprocess.check_call(["pip", "install", "torchdiffeq"])
    from torchdiffeq import odeint_adjoint as odeint

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure the device.

In [ ]:
set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Understanding ODEs: The Foundation

### What is an Ordinary Differential Equation?

An **ODE** describes how a quantity changes over time. The general form is:

$$\frac{dh(t)}{dt} = f(h(t), t)$$

Where:
- $h(t)$ is the state at time $t$ (like a hidden representation)
- $\frac{dh(t)}{dt}$ is the rate of change (derivative)
- $f(h(t), t)$ is a function that determines how the state changes

**Intuition**: Instead of saying "what is the state at the next time step", ODEs say "how is the state currently changing".

Let's visualize a simple ODE: $\frac{dh}{dt} = -0.3h$ (exponential decay).

In [ ]:
# Simple ODE: dh/dt = -0.3 * h
def simple_ode(t, h):
    return -0.3 * h

# Analytical solution: h(t) = h(0) * e^(-0.3*t)
t = np.linspace(0, 10, 100)
h_analytical = 1.0 * np.exp(-0.3 * t)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(t, h_analytical, 'b-', linewidth=2)
plt.xlabel('Time (t)')
plt.ylabel('State h(t)')
plt.title('Solution: h(t) = e^(-0.3t)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
derivatives = simple_ode(t, h_analytical)
plt.plot(t, derivatives, 'r-', linewidth=2)
plt.xlabel('Time (t)')
plt.ylabel('dh/dt')
plt.title('Rate of Change: dh/dt = -0.3h')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial state: h(0) = {h_analytical[0]:.3f}")
print(f"Final state: h(10) = {h_analytical[-1]:.3f}")
print(f"The state decays exponentially over time.")

**Key insight**: The ODE $\frac{dh}{dt} = f(h, t)$ defines a **continuous transformation** of the state over time. We can think of this as a smooth flow through state space.

### Numerical ODE Solvers

Most ODEs don't have analytical solutions, so we use **numerical methods** to approximate them. The simplest method is the **Euler method**:

$$h(t + \Delta t) \approx h(t) + \Delta t \cdot f(h(t), t)$$

This says: "the next state is the current state plus a small step in the direction of the derivative."

Let's implement the Euler method and compare it to the analytical solution.

In [ ]:
def euler_method(f, h0, t_span, n_steps):
    """Solve ODE using Euler method."""
    t_start, t_end = t_span
    dt = (t_end - t_start) / n_steps
    
    t_values = [t_start]
    h_values = [h0]
    
    t = t_start
    h = h0
    
    for _ in range(n_steps):
        # Euler step: h_next = h + dt * f(t, h)
        h = h + dt * f(t, h)
        t = t + dt
        
        h_values.append(h)
        t_values.append(t)
    
    return np.array(t_values), np.array(h_values)

# Compare different step sizes
t_span = (0, 10)
h0 = 1.0

plt.figure(figsize=(12, 4))

for i, n_steps in enumerate([5, 20, 100]):
    plt.subplot(1, 3, i + 1)
    
    # Euler approximation
    t_euler, h_euler = euler_method(simple_ode, h0, t_span, n_steps)
    
    # Analytical solution
    t_true = np.linspace(0, 10, 200)
    h_true = np.exp(-0.3 * t_true)
    
    plt.plot(t_true, h_true, 'b-', linewidth=2, label='Analytical', alpha=0.6)
    plt.plot(t_euler, h_euler, 'ro-', linewidth=2, markersize=5, label='Euler')
    
    error = np.abs(h_euler[-1] - np.exp(-0.3 * t_euler[-1]))
    plt.title(f'Steps: {n_steps}\nError: {error:.4f}')
    plt.xlabel('Time')
    plt.ylabel('State')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observation**: More steps = better approximation. With enough steps, Euler converges to the true solution. This trade-off between accuracy and computation is central to Neural ODEs.

## 2. From ResNets to Neural ODEs

### ResNet as Discrete Dynamics

Recall that a **ResNet block** has the form:

$$h_{t+1} = h_t + f(h_t, \theta_t)$$

Where $h_t$ is the hidden state at layer $t$, and $f$ is a function (typically Conv → BatchNorm → ReLU → Conv).

Compare this to the **Euler method**:

$$h(t + \Delta t) = h(t) + \Delta t \cdot f(h(t), t)$$

**Insight**: A ResNet is just the Euler method with $\Delta t = 1$!

Let's visualize this connection by implementing a simple ResNet and showing how it approximates a continuous transformation.

In [ ]:
class SimpleResBlock(nn.Module):
    """A simple residual block: h_next = h + f(h)"""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, dim),
        )
    
    def forward(self, h):
        # ResNet update: h_next = h + f(h)
        return h + self.net(h)

# Create a 10-layer ResNet
dim = 2
depth = 10
resnet = nn.Sequential(*[SimpleResBlock(dim) for _ in range(depth)])

# Trace the transformation through the network
h = torch.tensor([[1.0, 0.0]])  # Start point
trajectory = [h.detach().numpy()[0]]

with torch.no_grad():
    for block in resnet:
        h = block(h)
        trajectory.append(h.numpy()[0])

trajectory = np.array(trajectory)

plt.figure(figsize=(8, 6))
plt.plot(trajectory[:, 0], trajectory[:, 1], 'bo-', linewidth=2, markersize=8)
plt.scatter(trajectory[0, 0], trajectory[0, 1], c='green', s=200, marker='*', 
            zorder=5, label='Start')
plt.scatter(trajectory[-1, 0], trajectory[-1, 1], c='red', s=200, marker='*', 
            zorder=5, label='End')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title(f'ResNet Trajectory Through {depth} Layers\n(Each point is the output of one layer)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"Input: {trajectory[0]}")
print(f"Output: {trajectory[-1]}")
print(f"\nResNet creates a discrete path through state space.")

**Key insight**: ResNet defines a discrete sequence of transformations. What if we make the steps infinitesimally small and let depth approach infinity? We get a **continuous transformation** — a Neural ODE!

### The Continuous Limit

Taking the limit as $\Delta t \to 0$ in the Euler method:

$$\lim_{\Delta t \to 0} \frac{h(t + \Delta t) - h(t)}{\Delta t} = \frac{dh}{dt}$$

This gives us the **Neural ODE formulation**:

$$\frac{dh(t)}{dt} = f(h(t), t, \theta)$$

Where:
- $h(t)$ is the hidden state at continuous "depth" $t$
- $f$ is a neural network that computes the derivative
- $\theta$ are the parameters (shared across all "layers")

To compute the output, we **integrate** the ODE from $t=0$ to $t=1$:

$$h(1) = h(0) + \int_0^1 f(h(t), t, \theta) dt$$

Let's compare the discrete ResNet path with a continuous ODE path using the same dynamics function.

In [ ]:
# Define a simple dynamics function
def dynamics_func(t, h):
    """Define how the state changes: dh/dt = [-y, x] (rotation)"""
    x, y = h[0], h[1]
    return torch.tensor([[-y], [x]], dtype=torch.float32)

# Initial state
h0 = torch.tensor([[1.0], [0.0]])

# Discrete approximation (Euler with different step sizes)
def discrete_trajectory(h0, n_steps, t_end=2*np.pi):
    dt = t_end / n_steps
    trajectory = [h0.numpy().flatten()]
    h = h0.clone()
    
    for i in range(n_steps):
        t = i * dt
        dh = dynamics_func(t, h)
        h = h + dt * dh
        trajectory.append(h.numpy().flatten())
    
    return np.array(trajectory)

# Continuous solution using ODE solver
t_span = torch.linspace(0, 2*np.pi, 100)
h_continuous = odeint(dynamics_func, h0, t_span).squeeze().numpy()

# Compare different discretizations
plt.figure(figsize=(15, 4))

for i, n_steps in enumerate([4, 10, 50]):
    plt.subplot(1, 3, i + 1)
    
    # Discrete trajectory
    h_discrete = discrete_trajectory(h0, n_steps)
    
    # Plot
    plt.plot(h_continuous[:, 0], h_continuous[:, 1], 'b-', 
             linewidth=2, alpha=0.6, label='Continuous (ODE)')
    plt.plot(h_discrete[:, 0], h_discrete[:, 1], 'ro-', 
             markersize=6, linewidth=1.5, label=f'Discrete ({n_steps} steps)')
    
    plt.scatter(h0[0], h0[1], c='green', s=200, marker='*', zorder=5)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'{n_steps} Discrete Steps')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis('equal')

plt.tight_layout()
plt.show()

print("As the number of steps increases, the discrete approximation converges to the continuous solution.")
print("A ResNet with infinite depth would trace the smooth ODE trajectory!")

**Beautiful insight**: A ResNet with infinite layers and infinitesimal step size is exactly a Neural ODE. Neural ODEs are the continuous limit of residual networks!

## 3. Neural ODE Architecture

### The ODE Function

The core of a Neural ODE is the **ODE function** $f(h(t), t, \theta)$ that defines the dynamics. This is typically a neural network that:
- Takes the current hidden state $h(t)$ as input
- Optionally takes time $t$ as input
- Outputs the derivative $\frac{dh}{dt}$

Unlike ResNets where each layer has different parameters, Neural ODEs **share parameters** across all "depths" — the same function $f$ defines the dynamics everywhere.

Let's implement a simple ODE function that can be used in a Neural ODE.

In [ ]:
class ODEFunc(nn.Module):
    """The neural network that defines dh/dt."""
    
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64),
            nn.Tanh(),
            nn.Linear(64, dim),
        )
        
        # Track number of function evaluations (useful for understanding computational cost)
        self.nfe = 0
    
    def forward(self, t, h):
        """Compute dh/dt = f(h, t)
        
        Args:
            t: Current time (scalar or tensor)
            h: Hidden state of shape [batch_size, dim]
        
        Returns:
            dh/dt of shape [batch_size, dim]
        """
        self.nfe += 1
        return self.net(h)

# Test the ODE function
ode_func = ODEFunc(dim=10)
h = torch.randn(4, 10)  # Batch of 4 samples, dim=10
t = torch.tensor(0.5)

dh_dt = ode_func(t, h)

print(f"Input shape: {h.shape}")
print(f"Output (dh/dt) shape: {dh_dt.shape}")
print(f"\nThe ODE function takes a state and returns its derivative.")
print(f"This derivative tells us 'in which direction should the state change right now?'")

**Key point**: The ODE function doesn't compute the next state directly — it computes the **instantaneous rate of change**. The ODE solver uses this to integrate and find the next state.

### The Neural ODE Block

A **Neural ODE block** wraps the ODE function and uses an ODE solver to compute the output. The forward pass:

1. Start with input $h(0)$
2. Integrate from $t=0$ to $t=1$ using the ODE solver
3. Return $h(1)$

The magic: we use `odeint` from `torchdiffeq` which handles the integration and automatically computes gradients!

Let's implement the Neural ODE block.

In [ ]:
class ODEBlock(nn.Module):
    """A Neural ODE block that integrates the ODE function."""
    
    def __init__(self, odefunc):
        super().__init__()
        self.odefunc = odefunc
        self.integration_time = torch.tensor([0.0, 1.0])
    
    def forward(self, x):
        """Integrate the ODE from t=0 to t=1.
        
        Args:
            x: Input tensor of shape [batch_size, dim]
        
        Returns:
            Output tensor of shape [batch_size, dim]
        """
        # Move integration time to the same device as input
        self.integration_time = self.integration_time.type_as(x)
        
        # Solve the ODE: returns tensor of shape [2, batch_size, dim]
        # First element is h(0), second is h(1)
        out = odeint(self.odefunc, x, self.integration_time, rtol=1e-3, atol=1e-4)
        
        # Return only the final state h(1)
        return out[1]

# Test the ODE block
ode_func = ODEFunc(dim=10)
ode_block = ODEBlock(ode_func)

x = torch.randn(4, 10)
print(f"Input: {x.shape}")

# Reset NFE counter
ode_func.nfe = 0

# Forward pass
y = ode_block(x)
print(f"Output: {y.shape}")
print(f"\nNumber of function evaluations: {ode_func.nfe}")
print("The solver adaptively chooses how many times to evaluate the function!")

**Adaptive computation**: The ODE solver automatically decides how many function evaluations are needed based on the desired accuracy (controlled by `rtol` and `atol`). Simple regions need fewer evaluations; complex regions need more.

### Complete Neural ODE Classifier

Now let's build a complete classifier using Neural ODEs:

1. **Downsampling layer**: Convert input to feature space
2. **Neural ODE block**: Continuous transformation
3. **Classification head**: Global pooling + linear layer

Let's implement the complete Neural ODE classifier for MNIST.

In [ ]:
class ODEFuncImage(nn.Module):
    """ODE function for image features."""
    
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
        )
        self.nfe = 0
    
    def forward(self, t, x):
        self.nfe += 1
        return self.net(x)

class NeuralODEClassifier(nn.Module):
    """Image classifier using Neural ODE."""
    
    def __init__(self, num_classes=10, dim=64):
        super().__init__()
        
        # Downsampling: convert image to feature maps
        self.downsampling = nn.Sequential(
            nn.Conv2d(1, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
        )
        
        # Neural ODE block
        self.ode_func = ODEFuncImage(dim)
        self.ode_block = ODEBlock(self.ode_func)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )
    
    def forward(self, x):
        # Downsampling
        features = self.downsampling(x)
        
        # Continuous transformation via Neural ODE
        features = self.ode_block(features)
        
        # Classification
        logits = self.classifier(features)
        return logits

# Test the model
model = NeuralODEClassifier(num_classes=10, dim=32)
x = torch.randn(2, 1, 28, 28)

model.ode_func.nfe = 0
logits = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {logits.shape}")
print(f"Function evaluations: {model.ode_func.nfe}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

**Architecture**: The Neural ODE sits in the middle, providing a continuous transformation of features. The parameters are shared across all "depths" of the transformation!

## 4. Training on MNIST

### Data Preparation

Let's load MNIST and prepare dataloaders for training and validation.

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
])

# Load datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

Let's visualize some examples from the dataset.

In [ ]:
# Visualize samples
examples = next(iter(train_loader))
images, labels = examples

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i, 0].numpy(), cmap='gray')
    ax.set_title(f'Label: {labels[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Training Loop

Now let's train the Neural ODE classifier. We'll track accuracy, loss, and the number of function evaluations.

Let's implement the training and evaluation functions.

In [ ]:
def train_epoch(model, loader, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    total_nfe = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Reset NFE counter
        model.ode_func.nfe = 0
        
        # Forward pass
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
        total_nfe += model.ode_func.nfe
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.3f}',
            'acc': f'{100*correct/total:.1f}%',
            'nfe': model.ode_func.nfe
        })
    
    return total_loss / len(loader), 100 * correct / total, total_nfe / len(loader)

def evaluate(model, loader, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    total_nfe = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            
            # Reset NFE counter
            model.ode_func.nfe = 0
            
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            
            total_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            total_nfe += model.ode_func.nfe
    
    return total_loss / len(loader), 100 * correct / total, total_nfe / len(loader)

print("Training and evaluation functions ready!")

Now let's train the Neural ODE classifier. We'll train for just a few epochs to demonstrate the concept (feel free to increase epochs for better accuracy).

In [ ]:
# Create model
model = NeuralODEClassifier(num_classes=10, dim=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_nfe': [],
    'test_loss': [], 'test_acc': [], 'test_nfe': []
}

# Train
num_epochs = 3
print(f"Training Neural ODE for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Train
    train_loss, train_acc, train_nfe = train_epoch(model, train_loader, optimizer, device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_nfe'].append(train_nfe)
    
    # Evaluate
    test_loss, test_acc, test_nfe = evaluate(model, test_loader, device)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['test_nfe'].append(test_nfe)
    
    print(f"Train - Loss: {train_loss:.3f}, Acc: {train_acc:.2f}%, NFE: {train_nfe:.0f}")
    print(f"Test  - Loss: {test_loss:.3f}, Acc: {test_acc:.2f}%, NFE: {test_nfe:.0f}")

print("\nTraining complete!")

Let's visualize the training curves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0].plot(history['test_loss'], 'r-', label='Test', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(history['test_acc'], 'r-', label='Test', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# NFE (Function Evaluations)
axes[2].plot(history['train_nfe'], 'b-', label='Train', linewidth=2)
axes[2].plot(history['test_nfe'], 'r-', label='Test', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Avg NFE')
axes[2].set_title('Function Evaluations per Batch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy: {history['test_acc'][-1]:.2f}%")
print(f"Average function evaluations: {history['test_nfe'][-1]:.0f}")

**Observation**: The number of function evaluations (NFE) can vary during training! The ODE solver adapts based on the complexity of the dynamics.

## 5. Comparison with ResNet

### Building a Comparable ResNet

Let's build a ResNet with similar capacity and compare it to our Neural ODE.

Let's implement a simple ResNet classifier for comparison.

In [ ]:
class ResBlock(nn.Module):
    """Residual block: h_next = h + f(h)"""
    
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
        )
    
    def forward(self, x):
        return x + self.net(x)

class ResNetClassifier(nn.Module):
    """ResNet classifier for comparison."""
    
    def __init__(self, num_classes=10, dim=64, num_blocks=6):
        super().__init__()
        
        # Downsampling
        self.downsampling = nn.Sequential(
            nn.Conv2d(1, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.GroupNorm(min(32, dim), dim),
            nn.ReLU(inplace=True),
        )
        
        # Residual blocks
        self.blocks = nn.Sequential(*[ResBlock(dim) for _ in range(num_blocks)])
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )
        
        self.num_blocks = num_blocks
    
    def forward(self, x):
        features = self.downsampling(x)
        features = self.blocks(features)
        logits = self.classifier(features)
        return logits

# Compare parameter counts
ode_model = NeuralODEClassifier(num_classes=10, dim=32)
resnet_model = ResNetClassifier(num_classes=10, dim=32, num_blocks=6)

ode_params = sum(p.numel() for p in ode_model.parameters())
resnet_params = sum(p.numel() for p in resnet_model.parameters())

print(f"Neural ODE parameters: {ode_params:,}")
print(f"ResNet parameters: {resnet_params:,}")
print(f"\nResNet has {resnet_params / ode_params:.1f}x more parameters!")
print("Neural ODEs share parameters across 'depth', making them more parameter-efficient.")

**Key advantage**: Neural ODEs have far fewer parameters because they share the ODE function across all "depths". A 6-block ResNet has 6 separate sets of parameters!

Now let's train the ResNet for comparison.

In [ ]:
# Train ResNet
resnet = ResNetClassifier(num_classes=10, dim=32, num_blocks=6).to(device)
optimizer_resnet = torch.optim.Adam(resnet.parameters(), lr=1e-3)

resnet_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print(f"Training ResNet for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Train
    resnet.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc='Training ResNet'):
        images, labels = images.to(device), labels.to(device)
        
        logits = resnet(images)
        loss = F.cross_entropy(logits, labels)
        
        optimizer_resnet.zero_grad()
        loss.backward()
        optimizer_resnet.step()
        
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    
    train_loss = total_loss / len(train_loader)
    train_acc = 100 * correct / total
    resnet_history['train_loss'].append(train_loss)
    resnet_history['train_acc'].append(train_acc)
    
    # Test
    resnet.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing ResNet'):
            images, labels = images.to(device), labels.to(device)
            logits = resnet(images)
            loss = F.cross_entropy(logits, labels)
            
            total_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    
    test_loss = total_loss / len(test_loader)
    test_acc = 100 * correct / total
    resnet_history['test_loss'].append(test_loss)
    resnet_history['test_acc'].append(test_acc)
    
    print(f"Train - Loss: {train_loss:.3f}, Acc: {train_acc:.2f}%")
    print(f"Test  - Loss: {test_loss:.3f}, Acc: {test_acc:.2f}%")

print("\nResNet training complete!")

Let's compare the performance of Neural ODE vs ResNet.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test Loss
axes[0].plot(history['test_loss'], 'b-', linewidth=2, marker='o', label='Neural ODE')
axes[0].plot(resnet_history['test_loss'], 'r-', linewidth=2, marker='s', label='ResNet')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Loss')
axes[0].set_title('Test Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test Accuracy
axes[1].plot(history['test_acc'], 'b-', linewidth=2, marker='o', label='Neural ODE')
axes[1].plot(resnet_history['test_acc'], 'r-', linewidth=2, marker='s', label='ResNet')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Final Comparison ===")
print(f"Neural ODE - Acc: {history['test_acc'][-1]:.2f}%, Params: {ode_params:,}")
print(f"ResNet     - Acc: {resnet_history['test_acc'][-1]:.2f}%, Params: {resnet_params:,}")
print(f"\nNeural ODE achieves comparable accuracy with {resnet_params / ode_params:.1f}x fewer parameters!")

**Trade-offs**: Neural ODEs have fewer parameters but require more computation (multiple function evaluations). ResNets have more parameters but fixed computational cost.

## 6. Understanding the Adjoint Method

### Memory Efficiency Through Adjoints

One of the most important innovations in Neural ODEs is the **adjoint method** for computing gradients. 

**The problem**: Standard backpropagation through an ODE solver requires storing all intermediate states, leading to memory consumption that grows with the number of function evaluations.

**The solution**: The adjoint method computes gradients by solving a **backward ODE** that only needs $O(1)$ memory!

The adjoint state $a(t) = \frac{\partial L}{\partial h(t)}$ satisfies its own ODE:

$$\frac{da(t)}{dt} = -a(t)^T \frac{\partial f(h(t), t, \theta)}{\partial h}$$

Let's visualize the memory consumption of standard backprop vs adjoint method.

In [ ]:
# Simulate memory usage
nfe_values = np.arange(10, 101, 10)
memory_standard = nfe_values  # Memory grows linearly with NFE
memory_adjoint = np.ones_like(nfe_values) * 5  # Constant memory

plt.figure(figsize=(10, 5))
plt.plot(nfe_values, memory_standard, 'r-', linewidth=3, marker='o', 
         markersize=8, label='Standard Backprop')
plt.plot(nfe_values, memory_adjoint, 'b-', linewidth=3, marker='s', 
         markersize=8, label='Adjoint Method')
plt.xlabel('Number of Function Evaluations (NFE)')
plt.ylabel('Memory Consumption (Arbitrary Units)')
plt.title('Memory Efficiency: Standard Backprop vs Adjoint Method')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print("The adjoint method maintains constant memory regardless of depth!")
print("This is crucial for training very deep networks or using high-accuracy ODE solvers.")

**Revolutionary insight**: The adjoint method lets us train infinitely deep networks with constant memory. We used it automatically via `odeint_adjoint` from `torchdiffeq`!

## 7. Advantages and Applications

### Advantage 1: Adaptive Computation

Neural ODEs can **adapt their computational depth** based on the input. The ODE solver automatically uses more evaluations for complex inputs and fewer for simple ones.

Let's examine how NFE varies across different inputs.

In [ ]:
# Analyze NFE for different samples
model.eval()
sample_nfes = []
sample_images = []
sample_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        for i in range(len(images)):
            img = images[i:i+1].to(device)
            
            model.ode_func.nfe = 0
            _ = model(img)
            
            sample_nfes.append(model.ode_func.nfe)
            sample_images.append(images[i].numpy())
            sample_labels.append(labels[i].item())
            
            if len(sample_nfes) >= 100:
                break
        if len(sample_nfes) >= 100:
            break

sample_nfes = np.array(sample_nfes)

# Plot histogram
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(sample_nfes, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Number of Function Evaluations (NFE)')
plt.ylabel('Count')
plt.title('Distribution of NFE Across Test Samples')
plt.axvline(sample_nfes.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {sample_nfes.mean():.1f}')
plt.legend()
plt.grid(True, alpha=0.3)

# Show examples with min and max NFE
plt.subplot(1, 2, 2)
min_idx = sample_nfes.argmin()
max_idx = sample_nfes.argmax()

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample_images[min_idx][0], cmap='gray')
axes[0].set_title(f'Min NFE: {sample_nfes[min_idx]}\nLabel: {sample_labels[min_idx]}')
axes[0].axis('off')

axes[1].imshow(sample_images[max_idx][0], cmap='gray')
axes[1].set_title(f'Max NFE: {sample_nfes[max_idx]}\nLabel: {sample_labels[max_idx]}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"NFE statistics:")
print(f"  Min: {sample_nfes.min()}")
print(f"  Mean: {sample_nfes.mean():.1f}")
print(f"  Max: {sample_nfes.max()}")
print(f"  Std: {sample_nfes.std():.1f}")
print(f"\nThe model adapts its computational depth per input!")

**Adaptive depth**: Different inputs require different amounts of computation. Neural ODEs automatically adjust!

### Advantage 2: Continuous Representations

Since Neural ODEs define a continuous transformation, we can **evaluate at any time** $t$, not just discrete layers. This enables smooth interpolation and continuous-time modeling.

Let's visualize how a hidden state evolves continuously through the Neural ODE.

In [ ]:
# Trace continuous evolution
model.eval()

# Get a sample
sample_img, sample_label = test_dataset[0]
sample_img = sample_img.unsqueeze(0).to(device)

# Get features after downsampling
with torch.no_grad():
    features = model.downsampling(sample_img)
    
    # Trace evolution at multiple time points
    time_points = torch.linspace(0, 1, 20).to(device)
    trajectory = odeint(model.ode_func, features, time_points, rtol=1e-3, atol=1e-4)
    
    # Reduce dimensionality for visualization (take mean across spatial dims)
    trajectory_reduced = trajectory.mean(dim=[3, 4]).squeeze()  # [time, channels]

# Plot evolution of first 8 channels
plt.figure(figsize=(12, 6))

for i in range(8):
    plt.plot(time_points.cpu(), trajectory_reduced[:, i].cpu(), 
             linewidth=2, alpha=0.7, label=f'Channel {i}')

plt.xlabel('Time (Depth)', fontsize=12)
plt.ylabel('Feature Value', fontsize=12)
plt.title('Continuous Evolution of Features Through Neural ODE', fontsize=14)
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

print("Features evolve smoothly and continuously through the network!")
print("We can evaluate at ANY time t ∈ [0, 1], not just discrete layers.")

**Continuous depth**: Unlike discrete ResNets where we can only access layer outputs, Neural ODEs give us the entire continuous trajectory!

### Applications Beyond Classification

Neural ODEs have enabled breakthrough applications:

1. **Continuous Normalizing Flows (CNF)**: Generative models that learn probability distributions by transforming simple distributions through continuous flows. Unlike discrete normalizing flows, CNFs have unrestricted architectures and exact likelihood computation.

2. **Time Series Modeling**: Natural fit for irregularly-sampled time series. Can evaluate at any time point and handle missing data elegantly.

3. **Physical System Modeling**: When the underlying system is governed by differential equations (physics, chemistry, biology), Neural ODEs provide a natural parameterization.

4. **Latent ODEs**: Combine ODEs with VAEs to model latent dynamics in sequential data, enabling better long-term predictions.

Let's demonstrate a simple continuous normalizing flow concept: transforming a simple distribution.

In [ ]:
# Simple 2D example: transform a Gaussian through an ODE
class SimpleDynamics(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 2),
        )
    
    def forward(self, t, z):
        return self.net(z)

# Create samples from a simple Gaussian
z0 = torch.randn(500, 2)

# Transform through ODE
dynamics = SimpleDynamics()
time_points = torch.tensor([0.0, 1.0])

with torch.no_grad():
    z_trajectory = odeint(dynamics, z0, time_points)
    z1 = z_trajectory[1]

# Plot transformation
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(z0[:, 0], z0[:, 1], alpha=0.5, s=10)
plt.title('Initial Distribution (t=0)\nSimple Gaussian')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(z1[:, 0], z1[:, 1], alpha=0.5, s=10, c='red')
plt.title('Transformed Distribution (t=1)\nAfter Continuous Flow')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Neural ODEs can transform distributions continuously!")
print("This is the foundation of Continuous Normalizing Flows for generative modeling.")

**Power of continuous flows**: Neural ODEs can learn complex transformations of distributions, enabling flexible generative models!

## 8. Limitations and Considerations

### Computational Cost

While Neural ODEs have constant memory and fewer parameters, they can be **computationally expensive** during training:

- Multiple function evaluations (NFE) required per forward pass
- NFE varies and can be unpredictable
- Backward pass requires solving another ODE

**Trade-off**: Parameters ↓, Memory ↓, but Compute time ↑

### Training Stability

Neural ODEs can be sensitive to:

1. **Solver tolerance**: Too loose → inaccurate gradients; too tight → expensive computation
2. **Stiff ODEs**: Some dynamics are hard to integrate numerically
3. **Gradient pathologies**: The adjoint method can suffer from numerical instabilities

**Solutions**: Careful hyperparameter tuning, specialized ODE solvers, regularization techniques

### When to Use Neural ODEs?

**Use Neural ODEs when:**
- Memory is constrained (very deep networks)
- You need continuous representations (time series, flows)
- The problem has natural ODE structure (physics-informed learning)
- Parameter efficiency is crucial

**Use standard networks when:**
- Speed is critical
- Discrete depth is sufficient
- Simple problems don't benefit from continuous modeling

## Key Takeaways

**Core Insights:**

1. **ResNets are Euler methods**: ResNet blocks approximate the Euler method for solving ODEs with step size Δt = 1. Taking the limit as depth → ∞ and Δt → 0 gives Neural ODEs.

2. **Continuous transformation**: Neural ODEs define smooth flows through state space via $\frac{dh}{dt} = f(h, t, \theta)$. We can evaluate at any continuous time point, not just discrete layers.

3. **Memory efficiency**: The adjoint method enables O(1) memory backpropagation regardless of depth, solving a backward ODE instead of storing all intermediate states.

4. **Adaptive computation**: ODE solvers automatically adjust the number of function evaluations based on dynamics complexity, providing adaptive depth per input.

5. **Parameter sharing**: Unlike ResNets where each layer has separate parameters, Neural ODEs share the ODE function across all depths, drastically reducing parameter count.

**Practical Advantages:**
- Constant memory regardless of "depth"
- Fewer parameters than equivalent ResNets
- Smooth interpolation between time points
- Natural fit for continuous-time problems

**Trade-offs:**
- Higher computational cost (multiple function evaluations)
- Variable and unpredictable NFE during training
- Potential numerical instabilities with stiff ODEs

**Applications:**
- Continuous Normalizing Flows for generative modeling
- Time series with irregular sampling
- Physics-informed neural networks
- Memory-constrained environments

Neural ODEs represent a beautiful unification of differential equations and deep learning, offering a principled mathematical framework for continuous-depth architectures!